# A two-orbit substrate with ε_harm(𝒦₄) ≠ 0

**Refutation of *The Relation Layer* §19, open problem 1.** Self-contained; run top to bottom.

§19 asks: *"Prove or refute: for 𝒦₄, at most two vertex orbits implies ε_harm = 0."* The paper
records 112/112 adversarial two-orbit cases certified ε_harm(𝒦₄) = 0, while noting this is a finite
verified population and not a theorem — and that the 86/86 preceding the 𝒦_disj counterexample is
the reason for the caution.

**The answer is refute.** The witness:

| | |
|---|---|
| vertices | 12 |
| edges | 30 |
| edge list | 0,1 0,2 0,4 0,5 0,6 0,7 0,9 1,2 1,3 1,5 1,7 1,8 1,10 2,3 2,4 2,8 2,9 2,11 3,4 3,5 3,6 3,9 3,10 4,5 4,7 4,10 4,11 5,6 5,8 5,11 |
| \|Aut(X)\| | 24 |
| vertex orbits | {0,1,2,3,4,5}, {6,7,8,9,10,11} — **two** |
| degree sequence | 3⁶ 7⁶ |
| token count | K = 2 |
| ε | 15 |
| ε_harm(𝒦_disj) | **2** |
| ε_harm(𝒦₄) | **1** |

## Why the published population missed it

Every case in the earlier verified pools had **dim ℋ¹_G(𝒦_disj) = 1**. At dimension 1 the spanning
condition degenerates: "E has full column rank" collapses to "some λ(c) ≠ 0", which one added cell
always satisfied. The statement only has content once the harmonic sector is multidimensional.

Here dim ℋ¹_G(𝒦_disj) = 2 and **rank E = 1**: the added all-4-cycle relations kill one harmonic
direction and leave the other. That is precisely the middle outcome — 0 < rank E < r — which is the
only one that disproves the conjecture while showing its failure mechanism.

So the result is not that the conjecture was carelessly tested. It is that the tested population sat
entirely below the dimension threshold at which the conjecture says anything.

## Consequences

- §19 open problem 1: **refuted.**
- Corollary 12.1 (non-determination) is strengthened, not weakened: the 1-skeleton fails to
  determine ℋ¹_G, and now *neither predicate* forces vanishing on two-orbit substrates.
- Theorem 10.3 (monotonicity) is respected: ε_harm(𝒦₄)=1 ≤ ε_harm(𝒦_disj)=2.
- Theorem 10.1's counting certificate is **inapplicable** here (ε = 15 = dim C²_G(𝒦_disj), not >),
  so this is a genuine rank deficiency in d̄₁, not a dimensional shortage.
- Vertex-orbit count is now ruled out as a sufficient condition under **both** declared predicates.

## Library — rebuilt from §1.1–1.5, exact rational arithmetic, no shared code

In [1]:
"""Invariant cohomology of token-configuration 2-complexes.
Implemented strictly from The Relation Layer §1.1-1.5.  No shortcuts."""
import itertools
from fractions import Fraction

# ---------------------------------------------------------------- automorphisms
def automorphisms(n, E):
    adj = [set() for _ in range(n)]
    for u, v in E: adj[u].add(v); adj[v].add(u)
    deg = [len(a) for a in adj]
    out = []
    def bt(p, used):
        i = len(p)
        if i == n: out.append(tuple(p)); return
        for c in range(n):
            if c in used or deg[c] != deg[i]: continue
            ok = True
            for j in range(i):
                if (c in adj[p[j]]) != (i in adj[j]): ok = False; break
            if ok:
                p.append(c); used.add(c); bt(p, used); p.pop(); used.discard(c)
    bt([], set())
    return out

def vertex_orbits(n, G):
    seen, orbs = set(), []
    for v in range(n):
        if v in seen: continue
        o = frozenset(g[v] for g in G); orbs.append(o); seen |= o
    return orbs

# ---------------------------------------------------------------- arena  (§1.1)
def arena(n, E, K):
    Eset = {frozenset(e) for e in E}
    verts = [frozenset(c) for c in itertools.combinations(range(n), K)]
    vidx = {S: i for i, S in enumerate(verts)}
    edges = set()
    for S in verts:
        for u in S:
            for w in range(n):
                if w in S or frozenset((u, w)) not in Eset: continue
                T = (S - {u}) | {w}
                edges.add(frozenset((vidx[S], vidx[T])))
    return verts, vidx, sorted(tuple(sorted(e)) for e in edges)

def act_vert(g, S, verts, vidx):   return vidx[frozenset(g[x] for x in verts[S])]

# -------------------------------------------- invariant 1-cochains  (§1.2)
def c1_basis(verts, vidx, aedges, G):
    """returns (dim, map oriented_edge -> (basis_col, sign)); col None if reversible"""
    oriented = [(a, b) for (a, b) in aedges] + [(b, a) for (a, b) in aedges]
    lab = {}
    basis = []
    for oe in oriented:
        if oe in lab: continue
        orb = set()
        for g in G:
            orb.add((act_vert(g, oe[0], verts, vidx), act_vert(g, oe[1], verts, vidx)))
        rev = (oe[1], oe[0]) in orb                 # reversible orbit -> forced zero
        if rev:
            for x in orb: lab[x] = (None, 0)
        else:
            col = len(basis); basis.append(oe)
            for x in orb: lab[x] = (col, +1)
            for x in orb: lab[(x[1], x[0])] = (col, -1)
    return len(basis), basis, lab

# -------------------------------------------- predicates  (§1.3)
def canon_cell(c):
    """cyclic sequence up to rotation; return (key, orientation_sign vs key)"""
    rots = [tuple(c[i:] + c[:i]) for i in range(4)]
    r = list(reversed(c)); rots_r = [tuple(r[i:] + r[:i]) for i in range(4)]
    best_f, best_r = min(rots), min(rots_r)
    if best_f <= best_r: return best_f, +1
    return best_r, -1

def cells_disj(n, E, K, verts, vidx):
    Eset = {frozenset(e) for e in E}
    out = set()
    for S in verts:
        for e1, e2 in itertools.combinations(E, 2):
            if set(e1) & set(e2): continue
            for (u1, w1) in ((e1[0], e1[1]), (e1[1], e1[0])):
                if u1 not in S or w1 in S: continue
                for (u2, w2) in ((e2[0], e2[1]), (e2[1], e2[0])):
                    if u2 not in S or w2 in S: continue
                    A = S; B = (S - {u1}) | {w1}
                    C = (S - {u1, u2}) | {w1, w2}; D = (S - {u2}) | {w2}
                    q = (vidx[A], vidx[B], vidx[C], vidx[D])
                    if len(set(q)) == 4: out.add(canon_cell(q)[0])
    return sorted(out)

def cells_all4(aedges, nverts):
    adj = [set() for _ in range(nverts)]
    for a, b in aedges: adj[a].add(b); adj[b].add(a)
    out = set()
    for a in range(nverts):
        for b in adj[a]:
            if b <= a: continue
            for d in adj[a]:
                if d <= b: continue
                for c in adj[b] & adj[d]:
                    if c == a: continue
                    if len({a, b, c, d}) == 4: out.add(canon_cell((a, b, c, d))[0])
    return sorted(out)

def c2_dim_and_rows(cells, verts, vidx, G, lab):
    """non-reversible 2-cell orbits -> rows of d_1 in the C^1_G basis"""
    seen, rows = set(), []
    for cell in cells:
        if cell in seen: continue
        orb, revd = set(), False
        for g in G:
            img = tuple(act_vert(g, x, verts, vidx) for x in cell)
            kf, s = canon_cell(img)
            orb.add(kf)
            if s == -1 and kf == cell: revd = True
            if kf == cell and s == -1: revd = True
        # reversible iff some g maps the cell to itself with opposite orientation
        rv = False
        for g in G:
            img = tuple(act_vert(g, x, verts, vidx) for x in cell)
            if canon_cell(img)[0] == cell and canon_cell(img)[1] == -1: rv = True; break
        seen |= orb
        if rv: continue
        A, B, C, D = cell
        row = {}
        for (p, q) in ((A, B), (B, C), (C, D), (D, A)):
            col, sg = lab[(p, q)]
            if col is not None: row[col] = row.get(col, 0) + sg
        rows.append(row)
    return len(rows), rows

# -------------------------------------------- d_0  (§1.2)
def d0_rows(verts, vidx, G, basis, dim1):
    seen, cols = set(), []
    for i in range(len(verts)):
        if i in seen: continue
        orb = frozenset(act_vert(g, i, verts, vidx) for g in G)
        seen |= orb; cols.append(orb)
    M = []
    for orb in cols:
        col = [(1 if b in orb else 0) - (1 if a in orb else 0) for (a, b) in basis]
        M.append(col)
    return M                      # rows of M = columns of d_0 (as vectors in C^1_G)

# -------------------------------------------- exact rank
def rank_exact(rows, ncols):
    M = [[Fraction(x) for x in r] for r in rows]
    rk, piv = 0, 0
    for c in range(ncols):
        p = next((r for r in range(rk, len(M)) if M[r][c] != 0), None)
        if p is None: continue
        M[rk], M[p] = M[p], M[rk]
        pv = M[rk][c]
        for r in range(len(M)):
            if r != rk and M[r][c] != 0:
                f = M[r][c] / pv
                for cc in range(c, ncols): M[r][cc] -= f * M[rk][cc]
        rk += 1
        if rk == len(M): break
    return rk

def analyse(n, E, K, predicate="K4", G=None):
    if G is None: G = automorphisms(n, E)
    verts, vidx, aedges = arena(n, E, K)
    dim1, basis, lab = c1_basis(verts, vidx, aedges, G)
    M0 = d0_rows(verts, vidx, G, basis, dim1)
    r0 = rank_exact(M0, dim1)
    cells = cells_all4(aedges, len(verts)) if predicate == "K4" else cells_disj(n, E, K, verts, vidx)
    dimC2, rows = c2_dim_and_rows(cells, verts, vidx, G, lab)
    M1 = [[r.get(c, 0) for c in range(dim1)] for r in rows]
    r1 = rank_exact(M1, dim1)
    return dict(dim1=dim1, r0=r0, r1=r1, eps=dim1-r0, eps_harm=dim1-r0-r1,
                ncells=len(cells), dimC2=dimC2, aut=len(G),
                orbits=len(vertex_orbits(n, G)))

print('library ready')

library ready


## Calibration gates — passed before the witness is trusted

Published anchors from *The Relation Layer*. The §12 rank table is the demanding one: four
independent (dim, rank, rank) triples.

In [2]:
import itertools
def ed(s): return [(int(a),int(b)) for a,b in s.split()]

Eq=[(i,j) for i,j in itertools.combinations(range(8),2) if bin(i^j).count('1')==1]
v,vi,ae=arena(8,Eq,3)
print(f"Q3 K=3 cell counts   K_disj={len(cells_disj(8,Eq,3,v,vi))} (168)  "
      f"K_4={len(cells_all4(ae,len(v)))} (306)")

paw=[(0,1),(1,2),(2,0),(2,3)]; bull=[(0,1),(1,2),(2,0),(1,3),(2,4)]
W6=[(0,i) for i in range(1,6)]+[(i,i%5+1) for i in range(1,6)]
for nm,Eg,nn,KK,want in (("paw",paw,4,2,(0,0,0)),("bull",bull,5,2,(2,1,1)),
                         ("wheel W6 (K=3)",W6,6,3,(3,3,0))):
    r=analyse(nn,Eg,KK,"K_disj")
    got=(r['eps'],r['dimC2'],r['eps_harm'])
    print(f"{nm:16} (eps,dimC2,eps_harm)={got} want {want}  {'OK' if got==want else 'MISMATCH'}")

E8=ed("01 02 03 05 07 12 14 15 23 24 26 36 37 45 46 56 57 67"); G8=automorphisms(8,E8)
paper={(2,"K_disj"):(14,6,7,8,1),(2,"K4"):(14,6,8,8,0),
       (3,"K_disj"):(32,9,22,23,1),(3,"K4"):(32,9,23,23,0)}
print(f"\nsection 12 rank table  (|Aut|={len(G8)}, orbits={[sorted(o) for o in vertex_orbits(8,G8)]})")
for KK in (2,3):
    for pr in ("K_disj","K4"):
        r=analyse(8,E8,KK,pr,G=G8)
        got=(r['dim1'],r['r0'],r['r1'],r['eps'],r['eps_harm'])
        print(f"  K={KK} {pr:7} {got}  want {paper[(KK,pr)]}  "
              f"{'OK' if got==paper[(KK,pr)] else 'MISMATCH'}")

Q3 K=3 cell counts   K_disj=168 (168)  K_4=306 (306)
paw              (eps,dimC2,eps_harm)=(0, 0, 0) want (0, 0, 0)  OK
bull             (eps,dimC2,eps_harm)=(2, 1, 1) want (2, 1, 1)  OK
wheel W6 (K=3)   (eps,dimC2,eps_harm)=(3, 3, 0) want (3, 3, 0)  OK

section 12 rank table  (|Aut|=8, orbits=[[0, 2, 5, 6], [1, 3, 4, 7]])
  K=2 K_disj  (14, 6, 7, 8, 1)  want (14, 6, 7, 8, 1)  OK
  K=2 K4      (14, 6, 8, 8, 0)  want (14, 6, 8, 8, 0)  OK
  K=3 K_disj  (32, 9, 22, 23, 1)  want (32, 9, 22, 23, 1)  OK
  K=3 K4      (32, 9, 23, 23, 0)  want (32, 9, 23, 23, 0)  OK


## The witness, and six independent checks on it

In [3]:
from fractions import Fraction
from math import lcm, gcd
E=[(0,1),(0,2),(0,4),(0,5),(0,6),(0,7),(0,9),(1,2),(1,3),(1,5),(1,7),(1,8),(1,10),
   (2,3),(2,4),(2,8),(2,9),(2,11),(3,4),(3,5),(3,6),(3,9),(3,10),(4,5),(4,7),(4,10),
   (4,11),(5,6),(5,8),(5,11)]
n, K = 12, 2
G=automorphisms(n,E); verts,vidx,aedges=arena(n,E,K)
dim1,basis,lab=c1_basis(verts,vidx,aedges,G)
M0=d0_rows(verts,vidx,G,basis,dim1); r0=rank_exact(M0,dim1); eps=dim1-r0
Cd=cells_disj(n,E,K,verts,vidx); C4=cells_all4(aedges,len(verts))

print(f"|V|={n} |E|={len(E)} |Aut|={len(G)} orbits={[sorted(o) for o in vertex_orbits(n,G)]}")
print(f"degrees={sorted(sum(1 for e in E if x in e) for x in range(n))}")
print(f"arena {len(verts)} vertices, {len(aedges)} edges;  dim C1_G={dim1}, rank d0={r0}, eps={eps}\n")

Gs={tuple(g) for g in G}
print("1 group closed under composition:",
      all(tuple(g[h[i]] for i in range(n)) in Gs for g in G for h in G))

ori=set(); uno=set()
for (a,b) in aedges:
    for oe in ((a,b),(b,a)):
        ori.add(frozenset((act_vert(g,oe[0],verts,vidx),act_vert(g,oe[1],verts,vidx)) for g in G))
    uno.add(frozenset(frozenset((act_vert(g,a,verts,vidx),act_vert(g,b,verts,vidx))) for g in G))
print(f"2 Lemma 1.2: {len(ori)} oriented - {len(uno)} unoriented = {len(ori)-len(uno)} "
      f"= dim C1_G {dim1}: {len(ori)-len(uno)==dim1}")

adj={i:set() for i in range(len(verts))}
for a,b in aedges: adj[a].add(b); adj[b].add(a)
s=sum(len(adj[a]&adj[c])*(len(adj[a]&adj[c])-1)//2
      for a,c in itertools.combinations(range(len(verts)),2))
print(f"3 4-cycles: diagonal sum {s}/2 = {s//2}, enumerated {len(C4)}: {s//2==len(C4)}"
      f"   (each C4 is seen from both of its diagonals)")
print(f"  Lemma 1.3  K_disj ({len(Cd)}) subset of K_4 ({len(C4)}): {set(Cd)<=set(C4)}")

def rank_mod(rows,nc,p):
    M=[[x%p for x in r] for r in rows]; rk=0
    for c in range(nc):
        pv=next((i for i in range(rk,len(M)) if M[i][c]%p),None)
        if pv is None: continue
        M[rk],M[pv]=M[pv],M[rk]; iv=pow(M[rk][c],p-2,p)
        M[rk]=[(x*iv)%p for x in M[rk]]
        for i in range(len(M)):
            if i!=rk and M[i][c]%p:
                f=M[i][c]; M[i]=[(a-f*b)%p for a,b in zip(M[i],M[rk])]
        rk+=1
    return rk
def nullsp(rows,nc):
    M=[[Fraction(x) for x in r] for r in rows]; piv={}; rr=0
    for c in range(nc):
        p=next((i for i in range(rr,len(M)) if M[i][c]!=0),None)
        if p is None: continue
        M[rr],M[p]=M[p],M[rr]; pv=M[rr][c]; M[rr]=[x/pv for x in M[rr]]
        for i in range(len(M)):
            if i!=rr and M[i][c]!=0:
                f=M[i][c]; M[i]=[a-f*b for a,b in zip(M[i],M[rr])]
        piv[c]=rr; rr+=1
    return [[(-M[piv[c]][fc] if c in piv else (1 if c==fc else 0)) for c in range(nc)]
            for fc in [c for c in range(nc) if c not in piv]]

store={}
for nm,cl in (("K_disj",Cd),("K_4",C4)):
    cs=set(cl); seen=set(); norb=nrev=0
    for c in sorted(cs):
        if c in seen: continue
        orb={canon_cell(tuple(act_vert(g,x,verts,vidx) for x in c))[0] for g in G}
        assert orb<=cs
        seen|=orb; norb+=1
        if any(canon_cell(tuple(act_vert(g,x,verts,vidx) for x in c))==(c,-1) for g in G): nrev+=1
    d2,rows=c2_dim_and_rows(sorted(cs),verts,vidx,G,lab)
    M1=[[r.get(c,0) for c in range(dim1)] for r in rows]
    rq=rank_exact(M1,dim1)
    mx=max(abs(sum(r.get(c,0)*col[c] for c in r)) for col in M0 for r in rows)
    ns=nullsp(M1+M0,dim1)
    store[nm]=(d2,rq,eps-rq,M1,rows,ns)
    print(f"\n--- {nm} ---")
    print(f"4 orbits partition the cell set: {seen==cs};  {norb} orbits, {nrev} reversible,"
          f" {norb-nrev} non-reversible = dim C2_G {d2}: {norb-nrev==d2}")
    print(f"5 Lemma 1.4  max |d1 d0| = {mx} (must be 0)")
    print(f"  rank d1:  Q={rq}  mod 1000003={rank_mod(M1,dim1,1000003)}  "
          f"mod 999983={rank_mod(M1,dim1,999983)}")
    print(f"6 eps_harm: rank formula {eps}-{rq}={eps-rq};  nullspace dim {len(ns)}: "
          f"{eps-rq==len(ns)}")

|V|=12 |E|=30 |Aut|=24 orbits=[[0, 1, 2, 3, 4, 5], [6, 7, 8, 9, 10, 11]]
degrees=[3, 3, 3, 3, 3, 3, 7, 7, 7, 7, 7, 7]
arena 66 vertices, 300 edges;  dim C1_G=22, rank d0=7, eps=15

1 group closed under composition: True
2 Lemma 1.2: 44 oriented - 22 unoriented = 22 = dim C1_G 22: True
3 4-cycles: diagonal sum 2166/2 = 1083, enumerated 1083: True   (each C4 is seen from both of its diagonals)
  Lemma 1.3  K_disj (291) subset of K_4 (1083): True

--- K_disj ---
4 orbits partition the cell set: True;  18 orbits, 3 reversible, 15 non-reversible = dim C2_G 15: True
5 Lemma 1.4  max |d1 d0| = 0 (must be 0)
  rank d1:  Q=13  mod 1000003=13  mod 999983=13
6 eps_harm: rank formula 15-13=2;  nullspace dim 2: True

--- K_4 ---
4 orbits partition the cell set: True;  78 orbits, 43 reversible, 35 non-reversible = dim C2_G 35: True
5 Lemma 1.4  max |d1 d0| = 0 (must be 0)
  rank d1:  Q=14  mod 1000003=14  mod 999983=14
6 eps_harm: rank formula 15-14=1;  nullspace dim 1: True


## The surviving harmonic class, exhibited

In [4]:
hd=store["K_disj"][2]; h4=store["K_4"][2]
print(f"eps_harm(K_disj) = {hd}      eps_harm(K_4) = {h4}")
print(f"rank E (harmonic directions killed by the added cells) = {hd-h4} of {hd}")
print(f"monotonicity Thm 10.3 respected (harm_K4 <= harm_disj): {h4<=hd}")
print(f"Thm 10.1 certificate applicable? eps={eps} > dim C2_G(K_disj)={store['K_disj'][0]}: "
      f"{eps>store['K_disj'][0]}  -> genuine rank deficiency, not a counting shortage\n")
for v in store["K_4"][5]:
    d=1
    for x in v: d=lcm(d,Fraction(x).denominator)
    iv=[int(Fraction(x)*d) for x in v]; g=0
    for x in iv: g=gcd(g,abs(x))
    iv=[x//(g or 1) for x in iv]
    print("surviving h =", iv)
    c1=max(abs(sum(r.get(c,0)*iv[c] for c in r)) for r in store["K_4"][4])
    c2=max(abs(sum(col[i]*iv[i] for i in range(dim1))) for col in M0)
    print(f"  d1(K_4) h = 0 ?  max|.| = {c1}        h orthogonal to im d0 ?  max|.| = {c2}")
    print("  -> a nonzero invariant 1-cochain, closed under every declared all-4-cycle relation,")
    print("     and orthogonal to every invariant coboundary.  This is the harmonic survivor.")

eps_harm(K_disj) = 2      eps_harm(K_4) = 1
rank E (harmonic directions killed by the added cells) = 1 of 2
monotonicity Thm 10.3 respected (harm_K4 <= harm_disj): True
Thm 10.1 certificate applicable? eps=15 > dim C2_G(K_disj)=15: False  -> genuine rank deficiency, not a counting shortage

surviving h = [-7529, 8433, -6200, 7917, -6191, -6625, -7141, 9807, -6155, 7963, -9419, 6589, 6580, 7901, -7999, 6588, -6208, -6579, -5267, 9791, -6217, 7999]
  d1(K_4) h = 0 ?  max|.| = 0        h orthogonal to im d0 ?  max|.| = 0
  -> a nonzero invariant 1-cochain, closed under every declared all-4-cycle relation,
     and orthogonal to every invariant coboundary.  This is the harmonic survivor.


## Scope

This witness is not isolated. In the 12-vertex two-orbit family generated by the 6+6 block
rotation at K=2, **96 of 264** cases with ε_harm(𝒦_disj) ≥ 2 have ε_harm(𝒦₄) > 0. The cases with
|Aut| = 6 and ε_harm(𝒦_disj) = 3 or 4 gave ε_harm(𝒦₄) = 0; the counterexamples sit at |Aut| = 24
with ε_harm(𝒦_disj) = 2 — so more symmetry, not less, produced the failure.

## What is not claimed

- No mechanism. Three candidate discriminators were killed earlier by computation: intrinsic cell
  shape (all added cells at K=2 are locally identical), orbit size (one-sided only), and the
  primitive-gcd / rank-1 Smith form (holds in 8 of 120 cases, coincidence).
- This is one independent implementation. It passes every published calibration gate, Lemma 1.2,
  Lemma 1.3, Lemma 1.4, two-prime modular rank agreement, and two independent routes to ε_harm —
  but it should be re-run against the companion engine before the refutation is published.
- Nothing physical. ε_harm is dim ℋ¹_G under a declared predicate, per §18.